# 공정 조건별 품질 패턴 도출을 통한 제어 전략 제안

- 성결대학교 주관 데이터사이언스 경진대회 출품 프로젝트
- 실제 기업 공정 데이터를 기반으로 수행
- 원본 데이터는 기업 및 공정 관련 민감 정보로 인해 공개하지 않음
- 본 노트북은 발표 자료와 분석 결과를 바탕으로 재구성한 포트폴리오용 분석 코드임

## 분석 목표
- 품질 기준 변수 선정
- 품질 그룹별 공정 변수 차이 비교
- 공정 변수 간 상관관계 분석
- 제어 전략 도출
제어 전략 도출

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import ttest_ind

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

## 1. 데이터 불러오기
Process Data와 QC Data를 불러온다.

In [ ]:
DATA_DIR = Path("../data")

process_df = pd.read_excel(DATA_DIR / "Process_Data.xlsx")
qc_df = pd.read_excel(DATA_DIR / "QC_Data.xlsx")

process_df.columns = process_df.columns.str.strip()
qc_df.columns = qc_df.columns.str.strip()

print("Process Data shape:", process_df.shape)
print("QC Data shape:", qc_df.shape)

display(process_df.head())
display(qc_df.head())

## 2. 데이터 전처리 및 병합
날짜 기준으로 Process Data와 QC Data를 연결하기 전에 결측치, 날짜 형식, 이상치를 점검한다.

In [ ]:
# 실제 컬럼명에 맞게 수정 가능
process_date_col = "Date"
qc_date_col = "Date"
quality_col = "Weight"

# 날짜형 변환
process_df[process_date_col] = pd.to_datetime(process_df[process_date_col], errors="coerce")
qc_df[qc_date_col] = pd.to_datetime(qc_df[qc_date_col], errors="coerce")

# 날짜 컬럼 결측 확인
print("Process Date 결측 수:", process_df[process_date_col].isna().sum())
print("QC Date 결측 수:", qc_df[qc_date_col].isna().sum())

# 분석용 날짜 컬럼 생성 (일 단위)
process_df["merge_date"] = process_df[process_date_col].dt.date
qc_df["merge_date"] = qc_df[qc_date_col].dt.date

# 품질변수 결측 확인
print(f"{quality_col} 결측 수:", qc_df[quality_col].isna().sum())

# Weight 이상치 제거(IQR 방식)
q1 = qc_df[quality_col].quantile(0.25)
q3 = qc_df[quality_col].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

qc_filtered = qc_df[
    qc_df[quality_col].between(lower_bound, upper_bound, inclusive="both")
].copy()

print(f"{quality_col} 이상치 제거 전: {len(qc_df)}건")
print(f"{quality_col} 이상치 제거 후: {len(qc_filtered)}건")
print(f"IQR 범위: {lower_bound:.2f} ~ {upper_bound:.2f}")

# 날짜 중복 확인
print("\n[날짜 중복 확인]")
print("Process 일자별 건수 상위:")
display(process_df["merge_date"].value_counts().head())

print("QC 일자별 건수 상위:")
display(qc_filtered["merge_date"].value_counts().head())

# 같은 날짜에 여러 공정 측정치가 있을 수 있으므로 일자별 평균 집계 후 병합
numeric_process_cols = process_df.select_dtypes(include=[np.number]).columns.tolist()

process_daily = (
    process_df.groupby("merge_date")[numeric_process_cols]
    .mean()
    .reset_index()
)

qc_daily = (
    qc_filtered.groupby("merge_date")[[quality_col]]
    .mean()
    .reset_index()
)

merged_df = pd.merge(
    process_daily,
    qc_daily,
    on="merge_date",
    how="inner"
)

print("\nProcess Daily shape:", process_daily.shape)
print("QC Daily shape:", qc_daily.shape)
print("Merged Data shape:", merged_df.shape)

display(merged_df.head())

## 3. 품질 기준 설정
품질 지표로 `Weight`를 사용하고, 평균값을 기준으로 Good / Bad 그룹을 나눈다.

In [ ]:
quality_mean = merged_df[quality_col].mean()

merged_df["quality_group"] = np.where(
    merged_df[quality_col] >= quality_mean,
    "Good",
    "Bad"
)

print(f"{quality_col} 평균값: {quality_mean:.2f}")
print(merged_df["quality_group"].value_counts())

display(merged_df[["merge_date", quality_col, "quality_group"]].head())

## 4. 품질 그룹별 주요 공정 변수 비교
발표 자료에서 핵심적으로 다룬 `ReactA_Temp`, `ReactB_Temp`를 중심으로 그룹별 평균을 비교한다.

In [ ]:
key_process_vars = ["ReactA_Temp", "ReactB_Temp"]

group_mean = (
    merged_df.groupby("quality_group")[key_process_vars]
    .mean()
    .T
    .reset_index()
)

group_mean.columns = ["variable", "Bad", "Good"]
display(group_mean)

In [ ]:
ax = group_mean.set_index("variable")[["Bad", "Good"]].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("품질 그룹별 주요 공정 변수 평균 비교")
plt.xlabel("공정 변수")
plt.ylabel("평균값")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# boxplot으로 분포 비교
plot_df = merged_df.melt(
    id_vars=["quality_group"],
    value_vars=key_process_vars,
    var_name="variable",
    value_name="value"
)

plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_df, x="variable", y="value", hue="quality_group")
plt.title("품질 그룹별 주요 공정 변수 분포 비교")
plt.xlabel("공정 변수")
plt.ylabel("측정값")
plt.tight_layout()
plt.show()

## 5. 그룹별 차이 해석 및 통계 검정
Good / Bad 품질 그룹에서 어떤 공정 변수가 더 높게 나타나는지 확인하고, t-test로 평균 차이의 방향을 함께 점검한다.

In [ ]:
result_rows = []

for var in key_process_vars:
    good_data = merged_df.loc[merged_df["quality_group"] == "Good", var].dropna()
    bad_data = merged_df.loc[merged_df["quality_group"] == "Bad", var].dropna()

    good_mean = good_data.mean()
    bad_mean = bad_data.mean()

    t_stat, p_value = ttest_ind(good_data, bad_data, equal_var=False, nan_policy="omit")

    if good_mean > bad_mean:
        direction = "Good 그룹에서 더 높음"
    else:
        direction = "Bad 그룹에서 더 높음"

    result_rows.append({
        "variable": var,
        "Good_mean": round(good_mean, 3),
        "Bad_mean": round(bad_mean, 3),
        "Difference(Good-Bad)": round(good_mean - bad_mean, 3),
        "p_value": round(p_value, 4),
        "Interpretation": direction
    })

stats_result_df = pd.DataFrame(result_rows)
display(stats_result_df)

In [ ]:
for _, row in stats_result_df.iterrows():
    print(f"[{row['variable']}]")
    print(f" - Good 평균: {row['Good_mean']}")
    print(f" - Bad 평균: {row['Bad_mean']}")
    print(f" - 평균 차이(Good-Bad): {row['Difference(Good-Bad)']}")
    print(f" - p-value: {row['p_value']}")
    print(f" -> 해석: {row['Interpretation']}\n")

## 6. 공정 변수 간 상관관계 분석
공정 변수 간의 상관관계를 확인하여 중복 제어 가능성과 독립 제어 가능성을 탐색한다.

In [ ]:
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()
corr_df = merged_df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("공정 변수 간 상관관계 히트맵")
plt.tight_layout()
plt.show()

In [ ]:
# 핵심 변수 중심 상관관계 확인
focus_vars = ["ReactA_Temp", "ReactB_Temp", "ReactD_Temp", "ReactE_Temp", quality_col]
focus_vars = [col for col in focus_vars if col in corr_df.columns]

focus_corr = corr_df.loc[focus_vars, focus_vars]
display(focus_corr.round(3))

In [ ]:
# 상관계수 상위 쌍 추출
corr_pairs = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))
corr_pairs = (
    corr_pairs.stack()
    .reset_index()
)
corr_pairs.columns = ["var1", "var2", "correlation"]
corr_pairs["abs_corr"] = corr_pairs["correlation"].abs()

top_corr_pairs = corr_pairs.sort_values("abs_corr", ascending=False).head(10)
display(top_corr_pairs[["var1", "var2", "correlation"]])

## 7. 결론
- `Weight`를 품질 기준으로 설정하여 Good / Bad 그룹을 구분하였다.
- `ReactA_Temp`, `ReactB_Temp`는 품질 그룹별 차이를 보이는 핵심 공정 변수로 확인되었다.
- 그룹 평균 비교와 분포 비교를 통해 품질 차이에 영향을 줄 수 있는 공정 조건을 점검하였다.
- 상관관계 분석 결과, 일부 변수는 함께 움직이는 경향이 있어 중복 제어 최소화 가능성을 확인하였다.
- 향후에는 다변량 제어 및 머신러닝 기반 품질 예측 모델로 확장할 수 있다.